# Module 6: Semantic search engine

**Project:** build a search engine that finds documents **by meaning**, not by matching exact words. You embed your own documents, rank them against a question with cosine similarity, and measure how good the search really is.

**Done when:**
- you embedded **100 or more passages** from your own documents,
- your search returns a relevant passage in its **top 3 results for at least 15 of 20 test queries that you wrote**,
- you compared different chunk sizes and can explain which retrieves best.

**How to use this notebook**
1. Save your own copy (*File > Save a copy in Drive*). A free CPU runtime is enough: the embedding model is small.
2. Run cells from top to bottom. Fill in every `YOUR CODE HERE`. The **Self-check** cells tell you if you got it right.
3. Stuck? Use the hint ladder: docs, then the hints below each step, then the course forum. Only after your own attempt, open the solution page.
4. You will **reuse this index in Module 7** to build a chatbot, so keep your documents.

## Step 0: Setup (given)

In [ ]:
import re, time, json, random
from pathlib import Path
import numpy as np

try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"], check=True)
    from sentence_transformers import SentenceTransformer

random.seed(0); np.random.seed(0)
model = SentenceTransformer("all-MiniLM-L6-v2")      # a small, fast model that turns text into 384 numbers
print("Model loaded. Max input length:", model.max_seq_length, "tokens")

## Step 1: Your documents

Search needs something to search. Choose your own text: lecture notes, product manuals, FAQs, articles, a book chapter, or policies. Use **plain text or Markdown files** (`.txt` or `.md`).

- Put them in a folder named `my_docs` and upload it (zip it as `my_docs.zip` and upload that, or upload the files into a `my_docs` folder using the Files panel on the left).
- You need **at least about 6,000 words** in total, enough for 100 or more passages.
- Separate paragraphs with a **blank line**: the chunking step keeps paragraphs together.
- Prefer text where **different pages answer different questions**, so searching is meaningful.

**No documents yet?** Set `USE_DEMO = True` to build a demo collection from the documentation of 25 Python standard-library modules. You can then search it with questions like "pause the program for a few seconds".

In [ ]:
DOCS_DIR = Path("my_docs")
USE_DEMO = False      # True: build a demo collection instead of using your own documents

import inspect, importlib
DEMO_MODULES = ["re", "json", "math", "statistics", "random", "itertools", "collections", "datetime", "textwrap",
                "heapq", "bisect", "functools", "shutil", "csv", "urllib.parse", "os.path", "string", "fnmatch",
                "glob", "difflib", "zipfile", "base64", "hashlib", "time", "operator"]

def make_demo_docs(folder="my_docs"):
    """Write one text file per standard-library module, each holding the docstrings of its public functions and classes."""
    Path(folder).mkdir(exist_ok=True)
    total = 0
    for name in DEMO_MODULES:
        mod = importlib.import_module(name)
        label = name
        paras = []
        for attr, obj in sorted(vars(mod).items()):
            if attr.startswith("_") or not (inspect.isfunction(obj) or inspect.isclass(obj) or inspect.isbuiltin(obj)):
                continue
            doc = inspect.getdoc(obj)
            if not doc: continue
            doc = " ".join(doc.split())
            words = doc.split()
            if len(words) < 8: continue
            doc = " ".join(words[:60])
            paras.append(f"{label}.{attr}: {doc}")
        Path(folder, f"{name.replace('.', '_')}.txt").write_text("\n\n".join(paras), encoding="utf-8")
        total += len(paras)
    return total


if USE_DEMO:
    import shutil
    shutil.rmtree(DOCS_DIR, ignore_errors=True)
    print("Demo passages written:", make_demo_docs(str(DOCS_DIR)))
elif not DOCS_DIR.exists() and Path("my_docs.zip").exists():
    import zipfile
    with zipfile.ZipFile("my_docs.zip") as z:
        z.extractall(".")

def load_documents(folder):
    """Return a list of (file name, text) for every .txt and .md file in the folder."""
    docs = []
    for path in sorted(Path(folder).rglob("*")):
        if path.suffix.lower() in (".txt", ".md") and path.is_file():
            text = path.read_text(encoding="utf-8", errors="ignore").strip()
            if text:
                docs.append((path.name, text))
    return docs

assert DOCS_DIR.exists(), "Folder my_docs not found. Upload your documents (or set USE_DEMO = True) and run this cell again."
documents = load_documents(DOCS_DIR)
total_words = sum(len(t.split()) for _, t in documents)
print(f"{len(documents)} documents, {total_words:,} words")
for name, text in documents[:5]:
    print(f"  {name}: {len(text.split()):,} words")

### Self-check: documents

In [ ]:
assert len(documents) >= 1, "No .txt or .md files were found in my_docs."
assert total_words >= 6000, f"You have {total_words:,} words. Add more documents: you need about 6,000 words for 100+ passages."
print(f"Documents check passed: {len(documents)} documents, {total_words:,} words.")

## Step 2: Chunking

An embedding model reads a limited amount of text, and one vector for a long page blurs many topics together. So we cut documents into **chunks**: focused passages of limited size.

Write `chunk_text(text, max_words)`. It returns a list of chunk strings:
1. Split the text into **paragraphs** (separated by blank lines, `"\n\n"`), and ignore empty ones.
2. Pack **whole paragraphs** into a chunk until adding the next one would go over `max_words`. Then start a new chunk.
3. A paragraph that is **longer than `max_words` on its own** is cut into pieces of at most `max_words` words (it goes in its own chunks, and the chunk being built is closed first).

Join the paragraphs inside a chunk with `"\n\n"`.

*Hint:* keep a `current` list and a word `count`. `para.split()` gives the words of a paragraph.

In [ ]:
def chunk_text(text, max_words=60):
    """Group whole paragraphs into chunks of at most max_words words (split a too-long paragraph by words)."""
    chunks = []
    # YOUR CODE HERE
    return chunks

### Self-check: chunk_text

In [ ]:
_para = lambda n, tag: " ".join(f"{tag}{i}" for i in range(n))
_text = "\n\n".join([_para(30, "a"), _para(30, "b"), _para(30, "c"), _para(30, "d"), _para(200, "e")])
_ch = chunk_text(_text, max_words=70)
assert chunk_text("", 70) == [], "Empty text should give an empty list."
assert all(len(c.split()) <= 70 for c in _ch), "A chunk is longer than max_words."
assert " ".join(_ch).split() == _text.split(), "Chunks must contain all the words, in the original order, with nothing lost or repeated."
assert any("a0" in c.split() and "b29" in c.split() for c in _ch), "Two 30-word paragraphs fit in 70 words, so they should share a chunk. Pack whole paragraphs together."
assert not any("a0" in c.split() and "c0" in c.split() for c in _ch), "Three 30-word paragraphs are 90 words, more than 70. Start a new chunk before going over max_words."
assert len(_ch) == 5, f"Expected 5 chunks ([a+b], [c+d], and the 200-word paragraph in 3 pieces) but got {len(_ch)}."
print("chunk_text() check passed.")

Now build the chunks for all your documents. This cell is given.

In [ ]:
MAX_WORDS = 60

def build_index_texts(max_words):
    texts, sources = [], []
    for name, text in documents:
        for c in chunk_text(text, max_words):
            texts.append(c); sources.append(name)
    return texts, sources

chunk_texts, chunk_sources = build_index_texts(MAX_WORDS)
lengths = [len(c.split()) for c in chunk_texts]
print(f"{len(chunk_texts)} chunks of up to {MAX_WORDS} words (average {np.mean(lengths):.0f})")

### Self-check: chunks

In [ ]:
assert len(chunk_texts) >= 100, f"Only {len(chunk_texts)} chunks. You need at least 100: add more text, or use a smaller MAX_WORDS."
print(f"Chunks check passed: {len(chunk_texts)} passages ready to embed.")

## Step 3: Embeddings

An **embedding** turns a piece of text into a list of numbers (a vector), so that texts with similar meaning get similar vectors. The model you loaded gives 384 numbers per text.

Write `embed(texts)`. It takes a list of strings and returns a NumPy array of shape `(number of texts, 384)` where **every vector has length 1** (a unit vector).

*Hint:* `model.encode(list_of_texts, normalize_embeddings=True)` does it. Unit length makes cosine similarity simple, as you will see in Step 4.

In [ ]:
def embed(texts):
    """Turn a list of strings into an (n, d) NumPy array of unit-length vectors."""
    # YOUR CODE HERE
    return None

### Self-check: embed

In [ ]:
_e = embed(["A dog is playing in the garden", "A puppy plays outside", "The stock market fell sharply"])
assert isinstance(_e, np.ndarray) and _e.ndim == 2 and _e.shape[0] == 3, "embed() should return a 2-D NumPy array with one row per text."
assert np.allclose(np.linalg.norm(_e, axis=1), 1.0, atol=1e-3), "Each vector should have length 1. Pass normalize_embeddings=True."
_close, _far = float(_e[0] @ _e[1]), float(_e[0] @ _e[2])
assert _close > _far + 0.1, f"The dog and puppy sentences ({_close:.2f}) should be much closer than dog and stock market ({_far:.2f})."
print(f"embed() check passed. Dog vs puppy: {_close:.2f}   dog vs stock market: {_far:.2f}   (vector length {_e.shape[1]})")

Embed all your chunks. This cell is given.

In [ ]:
t0 = time.time()
chunk_embs = embed(chunk_texts)
print(f"Embedded {len(chunk_texts)} chunks in {time.time() - t0:.1f}s. Shape: {chunk_embs.shape}")

## Step 4: Cosine similarity

How do we say that two vectors are "close in meaning"? With the **cosine similarity**: the cosine of the angle between them. It is 1 when they point the same way (same meaning), about 0 when unrelated, and negative when opposite.

`cosine(a, b) = (a . b) / (length of a x length of b)`

Write `cosine_scores(query_vec, matrix)`. It takes one vector of shape `(d,)` and a matrix of shape `(n, d)`, and returns an array of `n` cosine similarities, one per row. Do **not** assume the vectors have length 1: divide by the lengths, so it works for any vectors. (Guard against division by zero with a tiny number such as `1e-12`.)

*Hint:* `matrix @ query_vec` gives all the dot products at once; `np.linalg.norm(matrix, axis=1)` gives the length of each row.

In [ ]:
def cosine_scores(query_vec, matrix):
    """Cosine similarity between one vector (d,) and every row of a matrix (n, d). Returns an array of n scores."""
    # YOUR CODE HERE
    return np.zeros(len(matrix))

### Self-check: cosine_scores

In [ ]:
_m = np.array([[1, 0], [0, 2], [3, 3], [-1, 0]], dtype=float)
_q = np.array([2, 0], dtype=float)
_s = cosine_scores(_q, _m)
assert _s.shape == (4,), f"Expected 4 scores, got shape {_s.shape}."
assert np.allclose(_s, [1.0, 0.0, 0.7071, -1.0], atol=1e-3), f"Scores should be [1, 0, 0.707, -1] but are {np.round(_s, 3).tolist()}. Remember to divide by the lengths of both vectors."
assert np.allclose(cosine_scores(_q * 5, _m * 7), _s, atol=1e-6), "Cosine similarity must not depend on vector length: scaling the vectors must not change the scores."
print("cosine_scores() check passed.")

## Step 5: Find the top matches

Write `top_k(query_vec, matrix, k)`. It returns two arrays: the **indices** of the `k` most similar rows, and their **scores**, both sorted from **most to least similar**. If `k` is larger than the number of rows, return all rows.

*Hint:* `np.argsort(scores)` sorts from lowest to highest, so you need to reverse it (`[::-1]`).

In [ ]:
def top_k(query_vec, matrix, k=3):
    """Return (indices, scores) of the k rows most similar to query_vec, best first."""
    # YOUR CODE HERE
    return np.array([], dtype=int), np.array([])

### Self-check: top_k

In [ ]:
_m = np.array([[1, 0], [0, 1], [0.9, 0.1], [-1, 0]], dtype=float)
_q = np.array([1, 0.05], dtype=float)
_idx, _sc = top_k(_q, _m, k=3)
assert list(_idx) == [0, 2, 1], f"Expected the best rows to be [0, 2, 1] in that order, got {list(_idx)}. Best first!"
assert len(_sc) == 3 and all(_sc[i] >= _sc[i + 1] for i in range(2)), "Scores must be sorted from highest to lowest."
assert len(top_k(_q, _m, k=10)[0]) == 4, "If k is larger than the number of rows, return all the rows."
print("top_k() check passed.")

Now put it together into a search function and try it. This cell is given.

In [ ]:
def search(query, k=3):
    """Embed the question and return the k most similar chunks."""
    idx, scores = top_k(embed([query])[0], chunk_embs, k)
    return [{"rank": r + 1, "score": float(s), "source": chunk_sources[i], "text": chunk_texts[i]}
            for r, (i, s) in enumerate(zip(idx, scores))]

def show(query, k=3):
    print(f"QUESTION: {query}")
    for hit in search(query, k):
        print(f"  #{hit['rank']}  score {hit['score']:.2f}  [{hit['source']}]")
        print("     " + hit["text"][:230].replace("\n", " ") + ("..." if len(hit["text"]) > 230 else ""))

show("pause the program for a few seconds" if USE_DEMO else "Write a question about your documents here")

**Experiment:** ask 5 questions of your own. Try rephrasing a question using *different words* than the document uses. Does the search still find it? (That is the whole point of semantic search.) Write what you noticed:

*Your notes:*

## Step 6: Measure how good your search is

Never trust a search engine by trying a few queries. Build a **test set** and count.

**Your test set:** write **at least 20 queries**, each with a list of `gold` phrases. A retrieved passage counts as **relevant** if it contains any of the gold phrases (ignoring upper and lower case). Rules for good test queries:
- Write the query **in your own words**, the way a real user would ask. Do **not** copy words from the document, or you only test keyword matching.
- Pick `gold` phrases that identify the right passage, such as a name or a distinctive phrase, and that **do not appear in the query**.
- Make sure the answer really is in your documents.

Format: `eval_set = [{"query": "your question", "gold": ["phrase that marks the right passage"]}, ...]`.

If you use the demo documents, a ready-made set called `DEMO_EVAL` is provided. Look at it to see the style, and write a few of your own as well.

In [ ]:
DEMO_EVAL = [
    {"query": "cut a string into pieces wherever a pattern occurs", "gold": ["re.split:"]},
    {"query": "average of a list of numbers", "gold": ["statistics.mean:", "statistics.fmean:"]},
    {"query": "find the value that occurs most often in data", "gold": ["statistics.mode:", "statistics.multimode:"]},
    {"query": "running totals of a sequence", "gold": ["itertools.accumulate:"]},
    {"query": "every possible ordering of the elements", "gold": ["itertools.permutations:"]},
    {"query": "count how often each item occurs", "gold": ["collections.Counter:"]},
    {"query": "turn a JSON text into Python data", "gold": ["json.loads:", "json.load:"]},
    {"query": "convert a Python object to text in JSON format", "gold": ["json.dumps:", "json.dump:"]},
    {"query": "read rows from a comma separated file", "gold": ["csv.reader:"]},
    {"query": "break long text into lines of a maximum width", "gold": ["textwrap.wrap:", "textwrap.fill:"]},
    {"query": "the few smallest values from a large dataset", "gold": ["heapq.nsmallest:"]},
    {"query": "add an element to a sorted list and keep it in order", "gold": ["bisect.insort"]},
    {"query": "copy a whole folder with everything inside it", "gold": ["shutil.copytree:"]},
    {"query": "compute a hash digest of some bytes", "gold": ["hashlib.md5:", "hashlib.sha1:", "hashlib.sha256:", "hashlib.sha512:"]},
    {"query": "how far apart are two moments in time", "gold": ["datetime.timedelta:"]},
    {"query": "does this file or folder exist", "gold": ["os.path.exists:"]},
    {"query": "break a web address into its parts", "gold": ["urllib.parse.urlparse:"]},
    {"query": "remember the results of expensive function calls", "gold": ["functools.lru_cache:"]},
    {"query": "find files whose names match a wildcard", "gold": ["glob.glob:", "fnmatch.fnmatch:"]},
    {"query": "pause the program for a few seconds", "gold": ["time.sleep:"]},
]

eval_set = []   # YOUR CODE HERE: at least 20 dictionaries {"query": ..., "gold": [...]}. With the demo documents you may start from DEMO_EVAL.
print(f"{len(eval_set)} test queries so far")

### Self-check: test set

In [ ]:
assert len(eval_set) >= 20, f"You have {len(eval_set)} test queries. Write at least 20."
_all_text = " ".join(chunk_texts).lower()
for _i, _item in enumerate(eval_set):
    assert isinstance(_item, dict) and _item.get("query") and _item.get("gold"), f"Test query {_i + 1} needs a 'query' and a non-empty 'gold' list."
    for _g in _item["gold"]:
        assert _g.lower() not in _item["query"].lower(), f"Gold phrase {_g!r} appears in its own query ({_item['query']!r}). Pick a phrase that is NOT in the question."
        assert _g.lower() in _all_text, f"Gold phrase {_g!r} (query {_i + 1}) is not in any chunk. Check the spelling, or make sure your documents contain the answer."
print(f"Test set check passed: {len(eval_set)} queries.")

Now write `evaluate_search(eval_set, texts, embs, embed_fn, k)`. It returns `(hits, missed)`:
- `hits`: the number of queries where **at least one of the top-k chunks contains at least one gold phrase** (ignore case),
- `missed`: the list of query strings that failed.

Use your own `top_k`. *Hint:* embed the query with `embed_fn([query])[0]`.

In [ ]:
def evaluate_search(eval_set, texts, embs, embed_fn, k=3):
    """Count the queries whose top-k chunks contain a gold phrase. Return (hits, list_of_missed_queries)."""
    hits, missed = 0, []
    # YOUR CODE HERE
    return hits, missed

### Self-check: evaluate_search

In [ ]:
def _toy_embed(texts):
    return np.array([[("cat" in t.lower()) + 0.01 * ("purr" in t.lower()) + 0.001,
                      ("dog" in t.lower()) + 0.01 * ("bark" in t.lower()) + 0.001] for t in texts], dtype=float)

_texts = ["Cats PURR softly", "dogs bark loudly", "cats nap all day", "birds sing"]
_set = [{"query": "tell me about cat", "gold": ["purr"]},
        {"query": "a dog story", "gold": ["BARK"]},
        {"query": "dog please", "gold": ["bark", "howl"]}]
_h1, _m1 = evaluate_search(_set, _texts, _toy_embed(_texts), _toy_embed, k=1)
_h2, _m2 = evaluate_search(_set, _texts, _toy_embed(_texts), _toy_embed, k=2)
assert _m1 == ["tell me about cat"] and _h1 == 2, (f"With k=1 the cat query is answered by 'cats nap all day' (no gold phrase), so it should be missed, "
    f"and the two dog queries are hits. Got hits={_h1}, missed={_m1}. Use the k that was passed in.")
assert _h2 == 3 and _m2 == [], (f"With k=2 all three queries should be hits (got {_h2}). Check two things: matching must ignore upper and lower case on BOTH sides "
    "(the chunk 'Cats PURR softly' must match 'purr', and 'BARK' must match 'bark'), and a query counts as a hit if ANY gold phrase is found, not all of them.")
print("evaluate_search() check passed.")

Run it on your search. This cell is given.

In [ ]:
hits, missed = evaluate_search(eval_set, chunk_texts, chunk_embs, embed, k=3)
print(f"top-3 hits: {hits} of {len(eval_set)}   (target: 15 or more)")
for q in missed:
    print("  MISSED:", q)

**Look at the misses.** For each missed query, run `show(query)` and decide: was the answer simply not in your documents, was the gold phrase too narrow, was the chunk cut badly, or did the model really fail? Write what you found:

*Your notes:*

## Step 7: Compare chunk sizes (fairly)

Chunk size is the biggest knob in a search engine. **Small chunks** are focused but may lose context. **Big chunks** hold more context but blur topics together.

**A fairness trap:** if every size returns its top 3, big chunks return far more text, so they are more likely to contain the answer by luck. To compare fairly, give every size the **same text budget**: about 360 words in total. So use `k = max(1, round(360 / size))`: 12 chunks of 30 words, 6 of 60, 3 of 120, but just 1 of 300.

Fill in the loop: for each chunk size, build the index with `build_index(size)` (given below), compute `k` from the budget, run `evaluate_search`, and store `{"chunks": ..., "k": ..., "hits": ...}` in `size_results[size]`.

In [ ]:
def build_index(max_words):
    """Chunk all documents at this size and embed them. Returns (texts, sources, embeddings)."""
    texts, sources = build_index_texts(max_words)
    return texts, sources, embed(texts)

size_results = {}
for size in [30, 60, 120, 300]:
    pass   # YOUR CODE HERE

for size, r in size_results.items():
    print(f"max {size:3d} words: {r['chunks']:4d} chunks, k={r['k']:2d} -> {r['hits']}/{len(eval_set)} hits")

### Self-check: chunk size comparison

In [ ]:
assert set(size_results) == {30, 60, 120, 300}, "size_results needs an entry for each of the sizes 30, 60, 120 and 300."
for _size, _r in size_results.items():
    assert _r["k"] == max(1, round(360 / _size)), f"For size {_size} the text budget gives k = {max(1, round(360 / _size))}, but you used k = {_r['k']}."
    assert 0 <= _r["hits"] <= len(eval_set), f"hits for size {_size} should be between 0 and {len(eval_set)}."
    assert _r["chunks"] > 0, f"No chunks were stored for size {_size}."
print("Chunk size comparison check passed.")

**Question:** which chunk size retrieved best at the equal text budget? Why do you think so? What would you expect to change if your documents had much longer paragraphs?

*Your answer:*

Now choose the best size. Set `BEST_WORDS` to the chunk size with the most hits (if tied, prefer the smaller size).

In [ ]:
BEST_WORDS = None    # YOUR CODE HERE: pick from size_results

### Self-check: best size

In [ ]:
assert BEST_WORDS in size_results, "Set BEST_WORDS to one of the sizes in size_results (30, 60, 120 or 300)."
_top = max(r["hits"] for r in size_results.values())
assert size_results[BEST_WORDS]["hits"] == _top, f"BEST_WORDS={BEST_WORDS} has {size_results[BEST_WORDS]['hits']} hits but another size reached {_top}."
print(f"Best chunk size: {BEST_WORDS} words.")

## Step 8: The final test

Rebuild the index with your best chunk size and run the real test: **top 3 results, at least 15 of your 20 queries**. This cell is given. It also saves your index, which you will reuse in Module 7.

In [ ]:
best_texts, best_sources, best_embs = build_index(BEST_WORDS)
final_hits, final_missed = evaluate_search(eval_set, best_texts, best_embs, embed, k=3)
print(f"chunk size {BEST_WORDS}: top-3 hits {final_hits} of {len(eval_set)}")
for q in final_missed:
    print("  MISSED:", q)

# save the index for Module 7 (a chatbot that answers from these documents)
np.save("index_embeddings.npy", best_embs)
Path("index_chunks.json").write_text(json.dumps({"texts": best_texts, "sources": best_sources}), encoding="utf-8")
print("Saved index_embeddings.npy and index_chunks.json")

assert final_hits >= 15, (f"Top-3 hits are {final_hits} of {len(eval_set)}, below the target of 15. "
                          "Look at the missed queries with show(query): fix gold phrases that are too narrow, try a different chunk size, or add documents that really contain the answers.")
print("Done! Your semantic search engine meets the target.")

## Stretch challenge (optional): re-rank

A fast first search can be refined by a slower, smarter second step. A **cross-encoder** reads the question and a passage *together* and scores how well the passage answers it. Use it to **re-rank**: retrieve the top 20 chunks with your embedding search, then let the cross-encoder pick the best 3.

```python
from sentence_transformers import CrossEncoder
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
scores = reranker.predict([(query, chunk) for chunk in candidate_chunks])
```

Measure the gain: run the same `eval_set` with and without re-ranking, and report the difference in top-3 hits. Is it worth the extra time per query? Write down what you found:

*Your notes:*

## Reflection and submission

Write three lines in your portfolio repo's `module-06/README.md`:
1. What documents you searched, and why
2. What surprised you about the results (a query that worked well, or one that failed)
3. Your best chunk size and your hit count

Then download this notebook (*File > Download > .ipynb*), push it to `module-06` in your portfolio repo (do **not** push private documents), and submit the link in Moodle.